# 03 · Train seq2seq (encoder-decoder + BPE) để so sánh với tagger

Cần **GPU T4**. Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → Change runtime type → T4 GPU.
Input: code + `data/splits` của notebook 01 (Colab: nằm sẵn trên Drive). Trên Colab mỗi lần train tối đa
150 phút, `runs/s2s/last.pt` lưu trên Drive, chạy lại notebook là train tiếp (`--resume`).
Cùng dữ liệu, số tham số tương đương (3 encoder + 3 decoder layer).
Mục đích: xem cách nhìn bài toán "dịch máy" có lợi/hại gì so với "gán nhãn ký tự":
chậm hơn bao nhiêu, và có bịa chữ (hallucination) không.

In [ ]:
import glob, json, os, sys
from pathlib import Path

IN_KAGGLE = os.path.exists("/kaggle/input")
IN_COLAB = "google.colab" in sys.modules
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/vi_diacritics/__init__.py", recursive=True)[0].rsplit("/vi_diacritics", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vi-diacritics-transformer" if IN_COLAB else os.path.abspath("vi-diacritics-transformer")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vi-diacritics-transformer {REPO}
    CODE = f"{REPO}/src"
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ai-portfolio/vi-diacritics-transformer" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE


def find(pat, hint):
    # output của notebook trước: Kaggle thì add làm input (/kaggle/input), Colab thì nằm sẵn trong WORK trên Drive
    for root in ("/kaggle/input", WORK, REPO):
        hits = glob.glob(f"{root}/{pat}") or glob.glob(f"{root}/**/{pat}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"không tìm thấy {pat}: {hint}")


SPLITS = os.path.dirname(find("splits/train.txt", "chạy notebook 01 trước"))
# Kaggle cho chạy liền 12h; Colab free hay ngắt sau vài tiếng nên mỗi lần chỉ chạy 150 phút,
# last.pt lưu trên Drive, chạy lại notebook là --resume train tiếp
MAX_MIN = 150 if IN_COLAB else 300
!pip install -q tokenizers

In [ ]:
!python -m vi_diacritics.train --task seq2seq --data {SPLITS} --out runs/s2s \
    --layers 6 --d-model 256 --heads 8 --d-ff 1024 --batch-size 128 --lr 5e-4 \
    --epochs 2 --eval-every 4000 --eval-n 1000 --max-minutes {MAX_MIN} --resume --bpe-vocab 8000

In [ ]:
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open("runs/s2s/log.jsonl", encoding="utf-8")]
tr = [r for r in rows if "loss" in r]
ev = [r["eval"] for r in rows if "eval" in r]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([r["step"] for r in tr], [r["loss"] for r in tr]); ax[0].set_title("train loss")
ax[1].plot([e["step"] for e in ev], [e["word_acc"] for e in ev], label="word acc")
ax[1].plot([e["step"] for e in ev], [e["hallucination_rate"] for e in ev], label="hallucination")
ax[1].legend()
fig.savefig("runs/s2s/curves.png", dpi=110, bbox_inches="tight")

## Vài ví dụ bịa chữ

In [ ]:
from vi_diacritics.dataset import read_lines
from vi_diacritics.evaluate import _seq2seq_fn
from vi_diacritics.labels import strip

f = _seq2seq_fn(Path("runs/s2s"))
test = read_lines(f"{SPLITS}/test.txt", 300)
bad = []
for g in test:
    p = f(strip(g))
    if strip(p) != strip(g):
        bad.append((strip(g), p))
print(f"{len(bad)}/{len(test)} câu bị đổi chữ")
for src, p in bad[:15]:
    print("IN :", src)
    print("OUT:", p)
    print()